# 점프 투 파이썬 Colab 실습 노트북

이 노트북은 [pahkey/jump2python](https://github.com/pahkey/jump2python) 공식 예제 저장소를 Colab으로 내려받아 장/절별로 탐색하고 실행하면서 공부하기 위한 실습용 노트북입니다.

핵심 흐름:

1. 공식 저장소를 `/content/jump2python`에 최신 상태로 다운로드합니다.
2. 다운로드된 전체 파일을 색인으로 만들고 누락 여부를 확인합니다.
3. 장/절 폴더를 선택해 예제 파일을 열어보고 실행합니다.
4. 원본 파일은 그대로 두고 `/content/my_jump2python_practice`에 복사해서 직접 수정합니다.
5. 필요하면 Google Drive에 내 실습 파일을 저장합니다.


## 1. 공식 저장소 다운로드

Colab 런타임이 초기화되면 `/content` 아래 파일은 사라질 수 있습니다. 그럴 때는 이 셀부터 다시 실행하면 됩니다.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import textwrap

REPO_URL = "https://github.com/pahkey/jump2python.git"
BRANCH = "main"
REPO_PATH = Path("/content/jump2python")
PRACTICE_PATH = Path("/content/my_jump2python_practice")

def run_cmd(args, cwd=None, check=True, show_output=True):
    result = subprocess.run(
        args,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if show_output and result.stdout:
        print(result.stdout.rstrip())
    if show_output and result.stderr:
        print(result.stderr.rstrip())
    if check and result.returncode != 0:
        raise RuntimeError(f"명령 실패: {args}\n{result.stderr}")
    return result

if REPO_PATH.exists() and (REPO_PATH / ".git").exists():
    run_cmd(["git", "fetch", "--depth=1", "origin", BRANCH], cwd=REPO_PATH)
    run_cmd(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_PATH)
else:
    if REPO_PATH.exists():
        shutil.rmtree(REPO_PATH)
    run_cmd(["git", "clone", "--depth=1", "--branch", BRANCH, REPO_URL, str(REPO_PATH)])

commit = run_cmd(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_PATH, show_output=False).stdout.strip()
print(f"다운로드 완료: {REPO_PATH}")
print(f"원본 저장소 커밋: {commit}")


## 2. 전체 파일 색인 만들기

아래 셀은 다운로드된 모든 파일을 표로 정리합니다. 원본 저장소의 Git 추적 파일 수와 실제 디스크에 있는 파일 수를 비교해 누락 여부도 확인합니다.


In [ ]:
import pandas as pd

def build_file_index(root=REPO_PATH):
    rows = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if ".git" in path.parts:
            continue
        rel = path.relative_to(root).as_posix()
        rows.append({
            "path": rel,
            "top": rel.split("/")[0],
            "extension": path.suffix or "(none)",
            "size": path.stat().st_size,
        })
    return pd.DataFrame(rows, columns=["path", "top", "extension", "size"])

file_index = build_file_index()
tracked_files = run_cmd(["git", "ls-tree", "-r", "--name-only", "HEAD"], cwd=REPO_PATH, show_output=False).stdout.splitlines()
missing_from_disk = sorted(set(tracked_files) - set(file_index["path"]))

print(f"Git 추적 파일: {len(tracked_files)}개")
print(f"디스크에서 확인한 파일: {len(file_index)}개")
print("누락 파일:", "없음" if not missing_from_disk else missing_from_disk)

display(file_index)


In [ ]:
summary = (
    file_index
    .assign(is_python=file_index["extension"].eq(".py"))
    .groupby("top", as_index=False)
    .agg(
        files=("path", "count"),
        python_files=("is_python", "sum"),
        bytes=("size", "sum"),
    )
    .sort_values("top")
)

display(summary)


In [ ]:
def print_tree(root=REPO_PATH, max_depth=3):
    root = Path(root)
    print(root.name + "/")
    for path in sorted(root.rglob("*")):
        rel = path.relative_to(root)
        if ".git" in rel.parts:
            continue
        if len(rel.parts) > max_depth:
            continue
        indent = "  " * (len(rel.parts) - 1)
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}{path.name}{suffix}")

print_tree(max_depth=3)


## 3. 장/절 폴더 선택

`chapter` 값을 바꾸면 해당 장/절의 예제 파일만 볼 수 있습니다. 예: `02-2`, `04-3`, `05-6`, `풀이`.


In [ ]:
chapters = sorted(file_index["top"].unique())
print("선택 가능한 상위 항목:")
print(", ".join(chapters))

chapter = "04-3"

if chapter == "README.md":
    chapter_files = file_index[file_index["path"].eq(chapter)]
else:
    chapter_files = file_index[file_index["path"].str.startswith(chapter + "/")]

if chapter_files.empty:
    print("해당 항목을 찾지 못했습니다. 위 목록에서 chapter 값을 다시 선택하세요.")
else:
    print(f"{chapter} 파일 수: {len(chapter_files)}개")
    display(chapter_files)


## 4. 파일 열어보기

원본 파일을 읽기 전용으로 확인합니다. 직접 수정하면서 실습하려면 다음 단계의 실습용 복사본을 사용하세요.


In [ ]:
def read_text_safely(path):
    path = Path(path)
    for encoding in ("utf-8", "cp949", "euc-kr"):
        try:
            return path.read_text(encoding=encoding), encoding
        except UnicodeDecodeError:
            pass
    return path.read_text(encoding="utf-8", errors="replace"), "utf-8(errors=replace)"

def show_local_file(path, start=1, end=None, title=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")
    text, encoding = read_text_safely(path)
    file_lines = text.splitlines()
    if end is None:
        end = len(file_lines)
    end = min(end, len(file_lines))
    label = title or str(path)
    print(f"{label} ({encoding}, {len(file_lines)} lines)")
    for line_no in range(start, end + 1):
        print(f"{line_no:>4}: {file_lines[line_no - 1]}")

def show_file(rel_path, start=1, end=None):
    return show_local_file(REPO_PATH / rel_path, start=start, end=end, title=rel_path)

selected_file = "04-3/read.py"
show_file(selected_file)


## 5. 예제 파일 실행하기

`.py` 파일을 선택해서 Colab 안에서 실행합니다. 일부 예제는 파일을 만들거나, 네트워크/브라우저/시간 지연을 사용하므로 실행 결과가 장마다 다를 수 있습니다.


In [ ]:
def run_python_file(rel_path, timeout=15):
    path = REPO_PATH / rel_path
    if path.suffix != ".py":
        raise ValueError(".py 파일만 실행할 수 있습니다.")
    if not path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {rel_path}")

    result = subprocess.run(
        [sys.executable, str(path)],
        cwd=path.parent,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    print(f"실행 파일: {rel_path}")
    print(f"종료 코드: {result.returncode}")
    print("\n[stdout]")
    print(result.stdout.rstrip() or "(없음)")
    print("\n[stderr]")
    print(result.stderr.rstrip() or "(없음)")
    return result

selected_file = "01-6/hello.py"
result = run_python_file(selected_file)


## 6. 키워드로 예제 찾기

책에서 배운 단어나 함수 이름으로 공식 예제 전체를 검색할 수 있습니다.


In [ ]:
def search_files(keyword, extensions=(".py", ".md", ".json"), limit=80):
    matches = []
    for rel_path in file_index["path"]:
        path = REPO_PATH / rel_path
        if path.suffix not in extensions:
            continue
        text, encoding = read_text_safely(path)
        for line_no, line in enumerate(text.splitlines(), start=1):
            if keyword.lower() in line.lower():
                matches.append({
                    "path": rel_path,
                    "line": line_no,
                    "text": line.strip(),
                    "encoding": encoding,
                })
                if len(matches) >= limit:
                    return pd.DataFrame(matches)
    return pd.DataFrame(matches, columns=["path", "line", "text", "encoding"])

keyword = "class"
search_result = search_files(keyword)
print(f"검색어: {keyword}")
display(search_result)


## 7. 실습용 복사본 만들기

원본 저장소 파일은 그대로 두고, 내가 수정할 파일은 `/content/my_jump2python_practice` 아래로 복사해서 사용합니다.


In [ ]:
def copy_for_practice(rel_path):
    src = REPO_PATH / rel_path
    if not src.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {rel_path}")
    dest = PRACTICE_PATH / rel_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)
    print(f"복사 완료: {dest}")
    return dest

practice_file = copy_for_practice("01-6/hello.py")
show_local_file(practice_file, title=str(practice_file))
print("\n실습 파일 경로:", practice_file)


In [ ]:
def list_practice_files():
    if not PRACTICE_PATH.exists():
        print("아직 실습용 복사본이 없습니다.")
        return []
    files = sorted(path.relative_to(PRACTICE_PATH).as_posix() for path in PRACTICE_PATH.rglob("*") if path.is_file())
    for file in files:
        print(file)
    return files

practice_files = list_practice_files()


## 8. Google Drive에 내 실습 파일 저장하기

Colab 런타임이 종료되어도 내 실습 파일을 남기고 싶을 때만 `SAVE_TO_DRIVE = True`로 바꿔 실행하세요.


In [ ]:
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive/jump2python_practice")
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree(PRACTICE_PATH, drive_dir, dirs_exist_ok=True)
    print(f"Google Drive에 저장 완료: {drive_dir}")
else:
    print("저장하지 않았습니다. 필요하면 SAVE_TO_DRIVE 값을 True로 바꾸세요.")


## 9. 실습 헬퍼: 간단 채점 함수

아래 함수는 내가 만든 코드가 예상 결과와 같은지 확인할 때 사용합니다.


In [ ]:
def check_equal(problem_name, actual, expected):
    if actual == expected:
        print(f"통과 - {problem_name}")
    else:
        print(f"다시 확인 - {problem_name}")
        print("   실제 결과:", actual)
        print("   예상 결과:", expected)

def check_true(problem_name, condition):
    if condition:
        print(f"통과 - {problem_name}")
    else:
        print(f"다시 확인 - {problem_name}")


## 10. 기초 실습 1: 자료형

아래 빈칸을 직접 채워보세요.


In [ ]:
# 문제 1
# 변수 a에 문자열 "Python"을 넣고, 변수 b에 정수 100을 넣으세요.

a = None
b = None

check_equal("문제 1-1", a, "Python")
check_equal("문제 1-2", b, 100)


In [ ]:
# 문제 2
# 리스트 numbers에서 숫자 3만 꺼내 result에 저장하세요.

numbers = [1, 2, 3, 4, 5]
result = None

check_equal("문제 2", result, 3)


In [ ]:
# 문제 3
# 딕셔너리 person에서 이름만 꺼내 name에 저장하세요.

person = {"name": "홍길동", "age": 30}
name = None

check_equal("문제 3", name, "홍길동")


## 11. 기초 실습 2: 조건문과 반복문


In [ ]:
# 문제 4
# score가 80 이상이면 "합격", 아니면 "불합격"을 result에 저장하세요.

score = 85
result = None

# 여기에 코드 작성

check_equal("문제 4", result, "합격")


In [ ]:
# 문제 5
# 1부터 10까지 숫자 중 짝수만 리스트 even_numbers에 담으세요.

even_numbers = []

# 여기에 코드 작성

check_equal("문제 5", even_numbers, [2, 4, 6, 8, 10])


## 12. 기초 실습 3: 함수


In [ ]:
# 문제 6
# 두 수를 더해서 반환하는 add 함수를 만드세요.

def add(a, b):
    pass

check_equal("문제 6-1", add(3, 4), 7)
check_equal("문제 6-2", add(10, 20), 30)


In [ ]:
# 문제 7
# 문자열을 입력받아 길이를 반환하는 get_length 함수를 만드세요.

def get_length(text):
    pass

check_equal("문제 7-1", get_length("python"), 6)
check_equal("문제 7-2", get_length("점프투파이썬"), 6)


## 13. 기초 실습 4: 파일 입출력

Colab에서는 현재 작업 폴더에 파일을 만들고 읽을 수 있습니다.


In [ ]:
# 문제 8
# memo.txt 파일에 "Hello Python"을 저장한 뒤 다시 읽어서 content에 저장하세요.

file_path = "/content/memo.txt"

# 여기에 코드 작성

content = None

# 여기에 코드 작성

check_equal("문제 8", content, "Hello Python")


## 14. 오류가 났을 때 GPT에게 물어보는 양식

아래 내용을 복사해서 GPT에게 보내면 됩니다.

```text
나는 점프 투 파이썬을 Colab에서 실습 중이야.
내가 하려는 것:
[여기에 목표 작성]

내 코드:
```python
[여기에 코드 붙여넣기]
```

오류 메시지:
[여기에 오류 메시지 붙여넣기]

정답을 바로 말하지 말고,
1단계 힌트, 2단계 힌트, 내가 요청하면 정답
순서로 도와줘.
```


## 15. 오늘 학습 정리 템플릿

공부가 끝나면 아래 내용을 채워보세요.

- 오늘 공부한 장/절:
- 실행해 본 공식 예제 파일:
- 오늘 새로 배운 개념 3개:
- 아직 헷갈리는 개념:
- 내가 직접 수정한 파일:
- 내일 복습할 내용:


In [ ]:
# 자유 실습 공간
# 여기에 책 내용을 따라 직접 코드를 입력하고 실행하세요.
